Validate attack.py against GPT-OSS and Gemma

|source|validation gpt_oos|validation gemma|validation score|Public LB|
|---|---|---|---|---|
|[Getting Started Notebook](https://www.kaggle.com/code/martynaplomecka/getting-started-notebook)|0.24|0.24|0.24|0.24

# Setup environment

In [1]:
import os, sys, json, time, subprocess, importlib.util, gc
from pathlib import Path

COMP_DIR = Path('/kaggle/input/competitions/ai-agent-security-multi-step-tool-attacks')
GPT_OSS_PATH = Path('/kaggle/input/models/llkh0a/gpt-oss-20b-gguf/pytorch/default/1/gpt_oss/gpt-oss-20b-Q4_K_M.gguf')
GEMMA_PATH = Path('/kaggle/input/models/llkh0a/gemma-4-26b-a4b-it-ud-q4-k-m-gguf/pytorch/default/1/gemma/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf')

WORK_DIR = Path('/kaggle/working/')
ARTIFACTS_DIR = WORK_DIR / 'artifacts'
ATTACK_PATH = WORK_DIR / 'attack.py'

WORK_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

assert COMP_DIR.exists(), f'Missing competition SDK: {COMP_DIR}'
assert GPT_OSS_PATH.exists(), f'Missing GPT-OSS GGUF: {GPT_OSS_PATH}'
assert GEMMA_PATH.exists(), f'Missing Gemma GGUF: {GEMMA_PATH}'

sys.path.insert(0, str(COMP_DIR))
os.environ['PYTHONUTF8'] = '1'
os.environ['GPT_OSS_MODEL_PATH'] = str(GPT_OSS_PATH)
os.environ['GEMMA_MODEL_PATH'] = str(GEMMA_PATH)

from kaggle_evaluation.jed_attack_134815 import jed_attack_gateway as gateway_defaults

BUDGET_S = gateway_defaults.DEFAULT_BUDGET_S          # 9000.0 in competition gateway
MAX_TOOL_HOPS = gateway_defaults.DEFAULT_MAX_TOOL_HOPS # 8 in competition gateway
ATTACK_SEED = gateway_defaults.ATTACK_SEED            # 123 in competition gateway
ENV_SELECTION_NAME = gateway_defaults.ENV_SELECTION   # "gym" in competition gateway
MODEL_NAMES = list(gateway_defaults.MODEL_NAMES)      # default: ["gpt_oss", "gemma"]

print('SDK:', COMP_DIR)
print('GPT_OSS_MODEL_PATH:', os.environ['GPT_OSS_MODEL_PATH'])
print('GEMMA_MODEL_PATH:', os.environ['GEMMA_MODEL_PATH'])
print('Work dir:', WORK_DIR)
print('Competition-matched settings:')
print(json.dumps({
    'budget_s_per_model': BUDGET_S,
    'max_tool_hops': MAX_TOOL_HOPS,
    'attack_seed': ATTACK_SEED,
    'env_selection': ENV_SELECTION_NAME,
    'model_names': MODEL_NAMES,
}, indent=2))
print('\nDisk status:')
subprocess.run(['df', '-h', '/kaggle/input', '/kaggle/working'], check=False)


SDK: /kaggle/input/competitions/ai-agent-security-multi-step-tool-attacks
GPT_OSS_MODEL_PATH: /kaggle/input/models/llkh0a/gpt-oss-20b-gguf/pytorch/default/1/gpt_oss/gpt-oss-20b-Q4_K_M.gguf
GEMMA_MODEL_PATH: /kaggle/input/models/llkh0a/gemma-4-26b-a4b-it-ud-q4-k-m-gguf/pytorch/default/1/gemma/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf
Work dir: /kaggle/working
Competition-matched settings:
{
  "budget_s_per_model": 9000.0,
  "max_tool_hops": 8,
  "attack_seed": 123,
  "env_selection": "gym",
  "model_names": [
    "gpt_oss",
    "gemma"
  ]
}

Disk status:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  132K   20G   1% /kaggle/input
/dev/loop1       20G  132K   20G   1% /kaggle/working


CompletedProcess(args=['df', '-h', '/kaggle/input', '/kaggle/working'], returncode=0)

## Install runtime dependency

Fresh Kaggle sessions may not include `llama-cpp-python`. This cell installs the CUDA wheel used by the GGUF model server, without extra GPU/debug output.


In [2]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('llama_cpp') is None:
    print('Installing llama-cpp-python CUDA wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--no-cache-dir',
        'llama-cpp-python',
        '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu124',
    ], check=True)

from llama_cpp import Llama
print('llama-cpp-python ready')


Installing llama-cpp-python CUDA wheel...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 328.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 20.5 MB/s eta 0:00:00
llama-cpp-python ready


In [12]:
from kaggle_evaluation.jed_attack_134815.gguf_model_server import GgufModelServer
from kaggle_evaluation.jed_attack_134815 import gpt_oss_model_server, gemma_model_server
from aicomp_sdk.evaluation.ops import build_attack_env
from aicomp_sdk.guardrails.optimal import Guardrail as OptimalGuardrail
from aicomp_sdk.core.env.api import EnvSelection
from aicomp_sdk.core.predicates import eval_predicates
import gc

# --- Model loading utilities ---
_current_model_name = None
_current_server = None

def load_model(model_name: str):
    """Load a model. Unloads previous if different."""
    global _current_model_name, _current_server

    if _current_model_name == model_name and _current_server is not None:
        print(f'Model {model_name} already loaded.')
        return _current_server

    if _current_server is not None:
        unload_model(_current_server, _current_model_name)
        _current_server = None
        _current_model_name = None

    if model_name == 'gpt_oss':
        spec = gpt_oss_model_server.SPEC
    elif model_name == 'gemma':
        spec = gemma_model_server.SPEC
    else:
        raise ValueError(f'Unknown model: {model_name}. Use "gpt_oss" or "gemma"')

    server = GgufModelServer(spec)
    print(f'Loading {model_name} GGUF...')
    t0 = time.time()
    server.load_model()
    print(f'Loaded {model_name} in {time.time() - t0:.1f}s')

    _current_model_name = model_name
    _current_server = server
    return server

def unload_model(server=None, label='model'):
    """Unload model and free GPU memory."""
    global _current_server, _current_model_name
    if server is None:
        server = _current_server
    if server is None:
        return
    try:
        server.unload()
    except Exception as err:
        print(f'{label} unload error:', repr(err))
    del server
    gc.collect()
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f'GPU after cleanup: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved')
    except Exception:
        pass
    _current_server = None
    _current_model_name = None


# --- Interactive test harness ---
class InteractiveTester:
    """Test single prompts interactively against a loaded model."""

    def __init__(self, model_name: str):
        self.model_name = model_name
        self.server = load_model(model_name)
        self.agent_factory = lambda: self.server._load_agent()
        self.env = None
        self._setup_env()

    def _setup_env(self):
        """Create fresh environment using build_attack_env (same as original notebook)."""
        self.env = build_attack_env(
            seed=ATTACK_SEED,
            fixtures_dir=COMP_DIR / 'aicomp_sdk' / 'fixtures',
            agent_factory=self.agent_factory,
            guardrail=OptimalGuardrail(),
            env_selection=EnvSelection.GYM,
            max_tool_hops=MAX_TOOL_HOPS,
        )
        self.env.reset()
        print(f'Environment ready for {self.model_name}')

    def reset(self):
        """Reset environment to clean state."""
        self.env.reset()
        print('Environment reset.')

    def chat(self, prompt: str, verbose: bool = True) -> dict:
        print(f'\n{"="*60}')
        print(f'PROMPT: {prompt}')
        print(f'{"="*60}')

        interact_result = self.env.interact(prompt)
        trace = self.env.export_trace_dict()
        predicates = eval_predicates(trace)

        output = {
            'agent_response': (trace.get('assistant_messages') or [''])[-1],
            'tool_calls': [],
            'tool_outputs': [],
            'predicates': predicates,
            'trace': trace,
        }

        tool_events = trace.get('tool_events', [])
        for event in tool_events:
            output['tool_calls'].append({
                'name': event.get('name'),
                'args': event.get('args'),
                'ok': event.get('ok'),
                'source': event.get('source'),
                'side_effect': event.get('side_effect'),
            })
            output['tool_outputs'].append({
                'output_preview': str(event.get('output', ''))[:500],
                'error': event.get('error'),
            })

        if verbose:
            self._print_result(output)
        return output

    def _print_result(self, output: dict):
        print(f'\n--- AGENT RESPONSE ---')
        resp = output['agent_response']
        print(resp[:2000] if len(resp) > 2000 else resp)

        print(f'\n--- TOOL CALLS ({len(output["tool_calls"])}) ---')
        for i, (call, out) in enumerate(zip(output['tool_calls'], output['tool_outputs'])):
            print(f'  [{i}] {call["name"]} | ok={call["ok"]} | side_effect={call["side_effect"]}')
            print(f'       args: {json.dumps(call["args"], indent=2)[:300]}')
            print(f'       output: {out["output_preview"][:200]}...')

        print(f'\n--- PREDICATES ---')
        if output['predicates']:
            print(json.dumps(output['predicates'], indent=2))
        else:
            print('No predicates triggered.')

        print(f'\n--- SUMMARY ---')
        print(f'Total tool calls: {len(output["tool_calls"])}')
        print(f'Predicates fired: {len(output["predicates"])}')

    def multi_turn(self, prompts: list[str], verbose: bool = True) -> list[dict]:
        results = []
        for prompt in prompts:
            result = self.chat(prompt, verbose=verbose)
            results.append(result)
            if result['predicates']:
                print(f'\n*** PREDICATES TRIGGERED! Stopping sequence. ***')
                break
        return results


def test(model_name: str = 'gpt_oss'):
    """Initialize interactive tester for a model."""
    global tester
    tester = InteractiveTester(model_name)
    return tester

print('Interactive test harness loaded.')
print('Usage:')
print('  tester = test("gpt_oss")   # or test("gemma")')
print('  result = tester.chat("your prompt here")')
print('  tester.reset()')
print('  results = tester.multi_turn(["prompt1", "prompt2"])')


✗ Option 1 failed: No module named 'aicomp_sdk.env'
✗ Option 2 failed: No module named 'aicomp_sdk.core.env.gym_env'
✓ Will use evaluate_redteam to create environment
Interactive test harness loaded.
Usage:
  tester = test("gpt_oss")   # or test("gemma")
  result = tester.chat("your prompt here")
  tester.reset()
  results = tester.multi_turn(["prompt1", "prompt2"])
ERROR! Session/line number was not unique in database. History logging moved to new session 29


In [13]:
# Load GPT-OSS and test a prompt
tester = test('gpt_oss')

# Now type prompts and see results:
result = tester.chat("Search the web for 'best restaurants'")

Loading gpt_oss GGUF...
Using pre-downloaded GGUF at: /kaggle/input/models/llkh0a/gpt-oss-20b-gguf/pytorch/default/1/gpt_oss/gpt-oss-20b-Q4_K_M.gguf


ValueError: Failed to load model from file: /kaggle/input/models/llkh0a/gpt-oss-20b-gguf/pytorch/default/1/gpt_oss/gpt-oss-20b-Q4_K_M.gguf

In [ ]:
# Check what predicates fired
print(json.dumps(result['predicates'], indent=2))

In [ ]:
from aicomp_sdk.evaluation.diagnostics import RunDiagnostics, EvaluatorVerbosity

def load_attack_class(path: Path):
    spec = importlib.util.spec_from_file_location('aas_dummy_attack', str(path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module.AttackAlgorithm

AttackAlgorithm = load_attack_class(ATTACK_PATH)

def compact_event(e):
    return {
        'name': e.get('name'),
        'ok': e.get('ok'),
        'source': e.get('source'),
        'side_effect': e.get('side_effect'),
        'args': e.get('args'),
        'output_preview': str(e.get('output', ''))[:240],
    }

def run_full_evaluation():
    """Run full competition evaluation against both models."""
    all_scores = {}
    
    for model_name in ['gpt_oss', 'gemma']:
        print(f"\n{'='*60}")
        print(f"EVALUATING: {model_name}")
        print(f"{'='*60}")
        
        # Unload previous model
        unload_model()
        
        agent_factory, model_server = build_gguf_agent_factory(model_name)
        
        transcript_file = ARTIFACTS_DIR / f'{model_name}_transcript.log'
        framework_file = ARTIFACTS_DIR / f'{model_name}_framework.jsonl'
        agent_debug_file = ARTIFACTS_DIR / f'{model_name}_agent-debug.jsonl'
        
        t0 = time.time()
        with RunDiagnostics(
            EvaluatorVerbosity.DEBUG,
            transcript_file=transcript_file,
            event_log_file=framework_file,
            agent_debug_file=agent_debug_file,
        ) as diagnostics:
            execution = evaluate_redteam(
                AttackAlgorithm,
                budget_s=BUDGET_S,
                agent_factory=agent_factory,
                agent_label=f'{model_name}_gguf',
                env_selection=EnvSelection.GYM,
                fixtures_dir=COMP_DIR / 'aicomp_sdk' / 'fixtures',
                diagnostics=diagnostics,
                attack_env_seed=ATTACK_SEED,
            )
        
        attack = execution.attack
        summary = {
            'model': model_name,
            'score_normalized_0_to_1000': attack.score,
            'score_raw': attack.score_raw,
            'findings_count': attack.findings_count,
            'unique_cells': attack.unique_cells,
            'unique_canonical_cells': attack.unique_canonical_cells,
            'evaluation_time_s': attack.time_taken,
            'wall_time_s': time.time() - t0,
            'guardrail_id': attack.guardrail_id,
            'guardrail_version': attack.guardrail_version,
        }
        
        summary_path = ARTIFACTS_DIR / f'{model_name}_summary.json'
        summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
        print(json.dumps(summary, indent=2))
        
        all_scores[f'{model_name}_public'] = float(attack.score)
        
        unload_model_server(model_server, model_name)
    
    all_scores['local_public_mean'] = sum(all_scores.values()) / 2
    print(f"\n{'='*60}")
    print('FINAL SCORES:')
    print(json.dumps(all_scores, indent=2))
    return all_scores

# Run when ready:
scores = run_full_evaluation()